# Python for Data Science — An Introduction for Actuaries

Welcome! This notebook is designed to get you productive in Python for data analysis. It assumes you're comfortable with probability distributions and regression (you are — you're an actuary), and that you've used R before. We'll build on that foundation.

**What we'll cover:**

| Part | Topic | What you'll learn |
|------|-------|-------------------|
| 1 | Python Syntax Essentials | Data types, collections, control flow, functions — the R-to-Python translation layer |
| 2 | Data Manipulation with Pandas | Loading, inspecting, filtering, grouping, transforming DataFrames |
| 3 | Visualisation with Matplotlib & Seaborn | Charts, distributions, and presentation-ready plots |
| 4 | Statistical Modelling with Statsmodels | OLS regression, Poisson GLMs for claim frequency modelling |

**The dataset:** We'll be working with the **freMTPL2freq** dataset — ~678,000 French motor third-party liability insurance policies. It's a classic actuarial dataset with claim counts, exposure periods, driver demographics, vehicle characteristics, and bonus-malus coefficients. You may recognise it from the CASdatasets R package.

---

## 0 — Environment Setup

Run the cell below to install the libraries we'll need. If you're in an environment where these are already installed, this will simply confirm that.

In [ ]:
# Install required packages (safe to re-run)
!pip install pandas numpy matplotlib seaborn statsmodels scikit-learn -q

In [ ]:
# Core imports — we'll explain each as we use them
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.datasets import fetch_openml

# Nicer defaults for plots
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

print("All imports successful — you're ready to go!")

---
# Part 1 — Python Syntax Essentials

If you know R, you already think in the right way. Python just spells things differently. This section covers the most important translations.

### 1.1 Variables & Basic Types

In R you use `<-` for assignment. In Python it's always `=`. Python is **dynamically typed** (like R) but there's no need for `c()` to make vectors — Python has its own collection types.

In [ ]:
# Numeric types
premium = 1250.75        # float (like R's numeric)
claim_count = 3           # int (R doesn't distinguish, Python does)
is_active = True          # bool (R: TRUE/FALSE, Python: True/False — note the capitalisation)

# Strings — single or double quotes, both work
policyholder = "Jane Doe"
region = 'Île-de-France'

# Check types with type()
print(type(premium))       # <class 'float'>
print(type(claim_count))   # <class 'int'>
print(type(is_active))     # <class 'bool'>

In [ ]:
# f-strings — Python's best feature for printing. Equivalent to R's glue() or paste0().
loss_ratio = 0.67
print(f"Policyholder: {policyholder}")
print(f"Loss ratio: {loss_ratio:.1%}")   # Format as percentage with 1 decimal

### 1.2 Collections: Lists, Tuples, Dictionaries

| Python | R equivalent | Mutable? | When to use |
|--------|-------------|----------|-------------|
| `list` | `list()` / `c()` | Yes | Ordered, changeable sequences |
| `tuple` | No direct equivalent | No | Fixed sequences (e.g., coordinates) |
| `dict` | Named list | Yes | Key-value lookups |
| `set` | No direct equivalent | Yes | Unique values, membership testing |

In [ ]:
# Lists — ordered, mutable. This is your R c() equivalent.
premiums = [1200, 980, 1550, 870, 2100]

# Indexing starts at 0 (NOT 1 like R — this WILL trip you up)
print(premiums[0])     # First element: 1200
print(premiums[-1])    # Last element: 2100
print(premiums[1:3])   # Slice — elements at index 1 and 2 (end is exclusive): [980, 1550]

In [ ]:
# List operations
premiums.append(1400)          # Add to end (R: c(premiums, 1400))
premiums.sort()                # Sort in place
print(f"Count: {len(premiums)}")  # len() is like R's length()
print(premiums)

In [ ]:
# Dictionaries — key-value pairs. Like R's named list.
policy = {
    "id": "POL-2024-001",
    "premium": 1250.75,
    "claims": 2,
    "vehicle_age": 5,
    "driver_age": 34,
}

# Access by key
print(policy["premium"])          # 1250.75
print(policy.get("deductible", 0))  # Returns 0 if key doesn't exist (safe access)

### 1.3 Control Flow

Python uses **indentation** instead of curly braces `{}`. This is non-negotiable — if your indentation is wrong, your code won't run.

In [ ]:
# if / elif / else  (R: if / else if / else)
bonus_malus = 120

if bonus_malus > 100:
    risk_class = "Malus (surcharge)"
elif bonus_malus == 100:
    risk_class = "Neutral"
else:
    risk_class = "Bonus (discount)"

print(f"BM coefficient {bonus_malus} → {risk_class}")

In [ ]:
# for loops — iterate over anything
regions = ["Île-de-France", "Bretagne", "Provence-Alpes-Côte d'Azur"]

for region in regions:
    print(f"Processing region: {region}")

# enumerate() gives you the index AND the value (like R's seq_along + indexing)
for i, region in enumerate(regions):
    print(f"  {i}: {region}")

In [ ]:
# range() — generates sequences of integers
# Equivalent to R's 1:5, but starts at 0 and the end is exclusive
for i in range(5):       # 0, 1, 2, 3, 4
    print(i, end=" ")

print()  # newline

for i in range(2, 8, 2):  # start=2, stop=8 (exclusive), step=2
    print(i, end=" ")     # 2, 4, 6

### 1.4 List Comprehensions

This is one of Python's most powerful features — compact, readable one-liners that replace explicit loops. Think of them as Python's version of R's `sapply()` or `purrr::map()`.

In [ ]:
# Basic: apply a formula to every element
premiums = [1200, 980, 1550, 870, 2100]

# Add 10% loading to each premium
loaded = [p * 1.10 for p in premiums]
print(loaded)

# With a filter: only premiums above 1000
high_premiums = [p for p in premiums if p > 1000]
print(high_premiums)

### 1.5 Functions

In R: `my_func <- function(x) { ... }`. In Python: `def my_func(x): ...`

In [ ]:
def burning_cost(claims: list, exposures: list) -> float:
    """
    Calculate the burning cost (pure premium) ratio.
    
    Parameters
    ----------
    claims : list of float
        Claim amounts per policy.
    exposures : list of float
        Exposure (in policy-years) per policy.
    
    Returns
    -------
    float
        Total claims divided by total exposure.
    """
    return sum(claims) / sum(exposures)


# Test it
sample_claims = [0, 0, 5000, 0, 12000]
sample_exposures = [1.0, 0.5, 1.0, 0.8, 1.0]

bc = burning_cost(sample_claims, sample_exposures)
print(f"Burning cost: {bc:,.2f} per policy-year")

In [ ]:
# Lambda functions — anonymous one-liners (like R's \(x) or function(x))
# Useful for quick transformations, especially inside pandas .apply()

to_percentage = lambda x: f"{x:.1%}"
print(to_percentage(0.6732))

### ✅ Quick Exercise — Part 1

Write a function called `classify_driver_age` that takes an integer `age` and returns:
- `"Young driver"` if age < 25
- `"Standard"` if 25 ≤ age < 65
- `"Senior"` if age ≥ 65

Then use a list comprehension to apply it to `ages = [19, 32, 45, 70, 23, 55]`.

In [ ]:
# YOUR CODE HERE



---
# Part 2 — Data Manipulation with Pandas

Pandas is Python's equivalent of R's `dplyr` + `tidyr`. The central object is the **DataFrame** — just like R's `data.frame` or `tibble`.

Let's load our dataset first.

### 2.1 Loading the freMTPL2freq Dataset

This dataset comes from the CASdatasets R package and is hosted on [OpenML](https://www.openml.org/d/41214). It contains **678,013** motor third-party liability policies from France.

**Columns:**

| Column | Description |
|--------|-------------|
| `IDpol` | Policy ID |
| `ClaimNb` | Number of claims during the exposure period |
| `Exposure` | Exposure period (fraction of a year) |
| `Area` | Area code (A through F — urban to rural) |
| `VehPower` | Vehicle power category (ordered, 4–15) |
| `VehAge` | Vehicle age in years |
| `DrivAge` | Driver age in years |
| `BonusMalus` | Bonus-malus coefficient (50–350; <100 = bonus, >100 = malus) |
| `VehBrand` | Vehicle brand group (anonymised A–G) |
| `VehGas` | Fuel type (Diesel / Regular) |
| `Density` | Population density (inhabitants/km²) of driver's city |
| `Region` | Policy region in France |

In [ ]:
# Fetch from OpenML (downloads ~30 MB on first run, cached after that)
from pathlib import Path

csv_path = Path("freMTPL2freq.csv")

if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"Loaded from local CSV: {len(df):,} rows")
else:
    data = fetch_openml(data_id=41214, as_frame=True)
    df = data.frame
    df.to_csv(csv_path, index=False)
    print(f"Downloaded and saved: {len(df):,} rows")

### 2.2 First Look at the Data

The R equivalents are in comments so you can build the mental mapping.

In [ ]:
# Shape: (rows, columns) — R: dim(df)
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

In [ ]:
# First 5 rows — R: head(df)
df.head()

In [ ]:
# Column types and non-null counts — R: str(df)
df.info()

In [ ]:
# Summary statistics for numeric columns — R: summary(df)
df.describe()

In [ ]:
# Check for missing values — R: colSums(is.na(df))
df.isnull().sum()

### 2.3 Data Types & Cleaning

The OpenML loader sometimes reads numeric columns as objects (strings). Let's enforce correct types.

In [ ]:
# Ensure numeric columns are numeric
numeric_cols = ["IDpol", "ClaimNb", "Exposure", "VehPower", "VehAge", "DrivAge", "BonusMalus", "Density"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Ensure categorical columns are category type (saves memory on 678K rows)
cat_cols = ["Area", "VehBrand", "VehGas", "Region"]
for col in cat_cols:
    df[col] = df[col].astype("category")

df.dtypes

### 2.4 Selecting & Filtering

This is where pandas starts to feel like `dplyr`.

In [ ]:
# Select columns — R: select(df, DrivAge, ClaimNb, Exposure)
df[["DrivAge", "ClaimNb", "Exposure"]].head()

In [ ]:
# Filter rows — R: filter(df, ClaimNb > 0)
claims_only = df[df["ClaimNb"] > 0]
print(f"Policies with ≥1 claim: {len(claims_only):,} ({len(claims_only)/len(df):.1%} of total)")

In [ ]:
# Multiple conditions — use & (and), | (or), ~ (not). Each condition in parentheses!
# R: filter(df, DrivAge < 25 & BonusMalus > 100)
young_malus = df[(df["DrivAge"] < 25) & (df["BonusMalus"] > 100)]
print(f"Young drivers with malus: {len(young_malus):,}")

### 2.5 Creating & Transforming Columns

Equivalent to R's `mutate()`.

In [ ]:
# Create a new column: annualised claim frequency
# R: mutate(df, Frequency = ClaimNb / Exposure)
df["Frequency"] = df["ClaimNb"] / df["Exposure"]

# Bin driver age into groups — R: cut()
bins = [17, 25, 35, 45, 55, 65, 100]
labels = ["18-25", "26-35", "36-45", "46-55", "56-65", "66+"]
df["AgeGroup"] = pd.cut(df["DrivAge"], bins=bins, labels=labels)

# Classify bonus-malus
df["BM_Class"] = pd.cut(
    df["BonusMalus"],
    bins=[0, 99, 100, 350],
    labels=["Bonus", "Neutral", "Malus"]
)

df[["DrivAge", "AgeGroup", "BonusMalus", "BM_Class", "Frequency"]].head(10)

### 2.6 Groupby & Aggregation

This is pandas' `group_by() |> summarise()`. It's one of the most important operations in data analysis.

In [ ]:
# Claim frequency by age group — R: df |> group_by(AgeGroup) |> summarise(...)
age_summary = (
    df.groupby("AgeGroup", observed=True)
    .agg(
        n_policies=("IDpol", "count"),
        total_claims=("ClaimNb", "sum"),
        total_exposure=("Exposure", "sum"),
    )
    .assign(frequency=lambda x: x["total_claims"] / x["total_exposure"])
)

age_summary

In [ ]:
# Multiple grouping variables
area_gas = (
    df.groupby(["Area", "VehGas"], observed=True)
    .agg(
        total_claims=("ClaimNb", "sum"),
        total_exposure=("Exposure", "sum"),
    )
    .assign(frequency=lambda x: x["total_claims"] / x["total_exposure"])
    .reset_index()  # Flatten the multi-index back into columns
)

area_gas.head(10)

In [ ]:
# Value counts — R: table() or count()
print("Area distribution:")
print(df["Area"].value_counts())
print()
print("Fuel type:")
print(df["VehGas"].value_counts(normalize=True))  # As proportions

### 2.7 Sorting

Equivalent to R's `arrange()`.

In [ ]:
# Top 10 highest-frequency policies
df.nlargest(10, "Frequency")[["IDpol", "DrivAge", "BonusMalus", "ClaimNb", "Exposure", "Frequency"]]

### ✅ Quick Exercise — Part 2

1. Filter the dataset to only **diesel** vehicles (`VehGas == 'Diesel'`) with **exposure ≥ 0.5**.
2. Group by `VehBrand` and calculate the total number of claims and total exposure for each brand.
3. Add a `frequency` column and sort by frequency descending.
4. Which vehicle brand group has the highest claim frequency among diesel vehicles?

In [ ]:
# YOUR CODE HERE



---
# Part 3 — Visualisation with Matplotlib & Seaborn

Matplotlib is Python's base plotting library (like R's base `plot()`). Seaborn sits on top of it and provides statistical plots with nicer defaults (like `ggplot2`).

The key mental model: **`plt`** (matplotlib) controls the figure/axes/layout, **`sns`** (seaborn) draws the statistical content.

### 3.1 Matplotlib Basics — The Figure & Axes Model

Every matplotlib plot has a **Figure** (the canvas) containing one or more **Axes** (individual plots). This is like R's `par(mfrow=...)` but more flexible.

In [ ]:
# Simple bar chart: claim frequency by age group
fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(age_summary.index.astype(str), age_summary["frequency"], color="steelblue", edgecolor="white")
ax.set_xlabel("Driver Age Group")
ax.set_ylabel("Claim Frequency (claims per policy-year)")
ax.set_title("Motor TPL Claim Frequency by Driver Age")

# Add value labels on top of each bar
for i, v in enumerate(age_summary["frequency"]):
    ax.text(i, v + 0.002, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Subplots: multiple charts in one figure
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: distribution of BonusMalus
axes[0].hist(df["BonusMalus"], bins=50, color="coral", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Bonus-Malus Coefficient")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Bonus-Malus")
axes[0].axvline(x=100, color="black", linestyle="--", label="Neutral (100)")
axes[0].legend()

# Right: distribution of Driver Age
axes[1].hist(df["DrivAge"], bins=50, color="teal", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Driver Age")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Driver Age")

plt.tight_layout()
plt.show()

### 3.2 Seaborn — Statistical Plots

Seaborn makes common statistical visualisations easy and good-looking out of the box.

In [ ]:
# Box plot: BonusMalus by Age Group
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="AgeGroup", y="BonusMalus", palette="viridis", ax=ax)
ax.set_title("Bonus-Malus Distribution by Driver Age Group")
ax.set_ylabel("Bonus-Malus Coefficient")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: claim frequency by Area × Fuel Type
pivot = area_gas.pivot(index="Area", columns="VehGas", values="frequency")

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(pivot, annot=True, fmt=".4f", cmap="YlOrRd", ax=ax)
ax.set_title("Claim Frequency: Area × Fuel Type")
plt.tight_layout()
plt.show()

In [ ]:
# Count plot: number of policies by region
fig, ax = plt.subplots(figsize=(12, 5))
region_order = df["Region"].value_counts().index
sns.countplot(data=df, x="Region", order=region_order, palette="mako", ax=ax)
ax.set_title("Policy Count by Region")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter with transparency — useful for large datasets
# Let's look at driver age vs. bonus-malus on a sample (plotting 678K points is slow)
sample = df.sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    sample["DrivAge"],
    sample["BonusMalus"],
    c=sample["ClaimNb"],
    cmap="coolwarm",
    alpha=0.4,
    s=10,
)
plt.colorbar(scatter, label="Claim Count")
ax.set_xlabel("Driver Age")
ax.set_ylabel("Bonus-Malus")
ax.set_title("Driver Age vs Bonus-Malus (5K sample, coloured by claim count)")
plt.tight_layout()
plt.show()

### ✅ Quick Exercise — Part 3

Create a **bar chart** showing the claim frequency by `VehBrand`. Use the grouped data from Part 2 or recalculate it. Add a horizontal reference line at the overall portfolio frequency.

*Hint: `ax.axhline(y=..., color='red', linestyle='--')` draws a horizontal line.*

In [ ]:
# YOUR CODE HERE



---
# Part 4 — Statistical Modelling with Statsmodels

Now for the part that connects to your actuarial training. We'll use **statsmodels**, which has a formula API that feels very similar to R's `glm()` syntax.

Our goal: **model claim frequency** (number of claims per unit of exposure) as a function of risk factors. This is the bread and butter of non-life pricing.

### 4.1 OLS Regression — A Familiar Starting Point

Let's start with ordinary least squares, which you already know. We'll regress the (log) claim frequency on driver age and bonus-malus.

Statsmodels has two APIs:
- **Formula API** (`smf`): uses R-style formulas like `"y ~ x1 + x2"` — start here!
- **Array API** (`sm`): pass numpy arrays directly — more control, less convenience.

In [ ]:
# Prepare a modelling subset: full-year exposures for cleaner frequency estimates
mod_df = df[df["Exposure"] >= 0.5].copy()
print(f"Modelling subset: {len(mod_df):,} policies")

# Quick OLS: frequency ~ DrivAge + BonusMalus
# R equivalent: lm(Frequency ~ DrivAge + BonusMalus, data = mod_df)
ols_model = smf.ols("Frequency ~ DrivAge + BonusMalus", data=mod_df).fit()

# The summary looks just like R's summary(model)
print(ols_model.summary())

In [ ]:
# Accessing specific results — like R's coef(), confint(), etc.
print("Coefficients:")
print(ols_model.params)
print()
print("R-squared:", round(ols_model.rsquared, 4))
print()
print("95% Confidence Intervals:")
print(ols_model.conf_int().rename(columns={0: "Lower", 1: "Upper"}))

### 4.2 Poisson GLM — The Right Tool for Claim Counts

OLS isn't ideal for count data. Claim counts follow a **Poisson** (or over-dispersed Poisson) distribution. In actuarial pricing, we model:

$$\text{ClaimNb}_i \sim \text{Poisson}(\mu_i), \quad \log(\mu_i) = \log(\text{Exposure}_i) + \mathbf{x}_i^\top \boldsymbol{\beta}$$

The `log(Exposure)` term is an **offset** — it adjusts for policies observed for different lengths of time.

This is exactly `glm(ClaimNb ~ ..., family = poisson, offset = log(Exposure))` in R.

In [ ]:
# Add the log-exposure offset
mod_df["log_exposure"] = np.log(mod_df["Exposure"])

# Poisson GLM with formula API
# R: glm(ClaimNb ~ DrivAge + BonusMalus + VehAge + VehGas + Area,
#        family = poisson(link = "log"), offset = log(Exposure), data = mod_df)

poisson_model = smf.glm(
    "ClaimNb ~ DrivAge + BonusMalus + VehAge + C(VehGas) + C(Area)",
    data=mod_df,
    family=sm.families.Poisson(),
    offset=mod_df["log_exposure"],
).fit()

print(poisson_model.summary())

**Reading the output:**
- Coefficients are on the **log scale** (link function = log). To get multiplicative effects, exponentiate them.
- `C(VehGas)` and `C(Area)` tell statsmodels to treat these as categorical variables (like R's `factor()`).
- The **deviance** and **Pearson chi-squared** help assess model fit.

In [ ]:
# Exponentiate coefficients → multiplicative risk factors
# e.g., exp(0.05) = 1.051 means a 5.1% increase in expected claim frequency
risk_factors = np.exp(poisson_model.params)
risk_factors_df = pd.DataFrame({
    "Coefficient": poisson_model.params,
    "Risk Factor (exp)": risk_factors,
    "p-value": poisson_model.pvalues,
})

print("Exponentiated coefficients (risk relativities):")
risk_factors_df.round(4)

### 4.3 Model Diagnostics

Let's check how well our Poisson model fits.

In [ ]:
# Deviance residuals
resid = poisson_model.resid_deviance

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs fitted
fitted = poisson_model.fittedvalues
axes[0].scatter(fitted, resid, alpha=0.05, s=2, color="steelblue")
axes[0].axhline(y=0, color="red", linestyle="--")
axes[0].set_xlabel("Fitted Values")
axes[0].set_ylabel("Deviance Residuals")
axes[0].set_title("Residuals vs Fitted")

# Distribution of residuals
axes[1].hist(resid, bins=80, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].set_xlabel("Deviance Residuals")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Deviance Residuals")

plt.tight_layout()
plt.show()

# Over-dispersion check: Pearson chi-squared / degrees of freedom
# If >> 1, the data is over-dispersed relative to Poisson
pearson_chi2 = poisson_model.pearson_chi2
dof = poisson_model.df_resid
dispersion = pearson_chi2 / dof
print(f"Pearson dispersion: {dispersion:.3f}")
print("(Values much > 1 suggest over-dispersion — consider Negative Binomial or quasi-Poisson)")

### 4.4 Predicted vs Actual Frequency by Age Group

A classic actuarial validation: aggregate predicted and actual frequencies by a key rating factor.

In [ ]:
# Add predictions to the modelling dataframe
mod_df["predicted_claims"] = poisson_model.fittedvalues

# Compare predicted vs actual by age group
comparison = (
    mod_df.groupby("AgeGroup", observed=True)
    .agg(
        actual_claims=("ClaimNb", "sum"),
        predicted_claims=("predicted_claims", "sum"),
        exposure=("Exposure", "sum"),
    )
    .assign(
        actual_freq=lambda x: x["actual_claims"] / x["exposure"],
        predicted_freq=lambda x: x["predicted_claims"] / x["exposure"],
    )
)

# Plot it
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(comparison))
width = 0.35

ax.bar([i - width/2 for i in x], comparison["actual_freq"], width, label="Actual", color="steelblue")
ax.bar([i + width/2 for i in x], comparison["predicted_freq"], width, label="Predicted", color="coral")

ax.set_xticks(x)
ax.set_xticklabels(comparison.index.astype(str))
ax.set_xlabel("Driver Age Group")
ax.set_ylabel("Claim Frequency")
ax.set_title("Actual vs Predicted Claim Frequency by Age Group")
ax.legend()
plt.tight_layout()
plt.show()

comparison[["actual_freq", "predicted_freq"]].round(4)

### ✅ Final Exercise — Part 4

Extend the Poisson GLM above by:

1. Adding `VehPower` and `Density` (try `np.log(Density)` since population density is heavily right-skewed) as additional predictors.
2. Fit the extended model.
3. Compare the AIC of the original and extended models. Which fits better?
4. Create an actual vs predicted comparison chart by `Area` instead of `AgeGroup`.

*Hint: AIC is available via `model.aic`. Lower AIC = better fit.*

In [ ]:
# YOUR CODE HERE



---
## Where to Go from Here

You've now covered the core Python data science stack. Here are natural next steps:

- **scikit-learn** — machine learning models (random forests, gradient boosting, cross-validation)
- **Negative Binomial GLM** — handles the over-dispersion you likely observed above (`sm.families.NegativeBinomial()`)
- **Tweedie models** — for aggregate loss modelling (combined frequency × severity)
- **Feature engineering** — interaction terms, splines, target encoding for high-cardinality categoricals
- **XGBoost / LightGBM** — modern actuarial pricing increasingly uses gradient-boosted trees alongside GLMs

The freMTPL2 dataset also has a companion **severity** dataset (`freMTPL2sev`, OpenML data_id 41215) with individual claim amounts — a natural extension for building a complete frequency × severity pricing model.